# Indicator Analysis — Freedom in the World

**Notebook 07 of 08**

### Purpose

Notebooks 04–06 worked mainly with the overall score and its subtotals. This notebook turns to the **40 indicators themselves**: what each measures, where the world stands on each in 2026, which dimensions changed most, and how the two special scales — the **1–7 ratings** (inverted) and the **categorical status** — behave.

The guiding question, from the project spec:

> Which specific dimensions of political rights and civil liberties are changing, rather than only whether overall freedom is changing?

## Data and Inputs

| Item | Location |
|---|---|
| Long analytical dataset | `data/processed/freedom_in_world_long.csv` |

**Question this notebook answers:** *What do the 40 indicators reveal, individually, about the level and direction of freedom in 2026?*

**Method:** build a full per-indicator profile (level + change), examine the category trends, then treat the ratings and the status classification separately — they are not on the 0–n scales and must not be mixed into them.

## Setup: imports and the project root

Same bootstrap as the previous notebooks.

In [1]:
import sys
from pathlib import Path

current = Path.cwd()
while not (current / 'data' / 'raw' / 'FH_FIW_WIDEF.csv').exists():
    current = current.parent
    if current == current.parent:
        raise RuntimeError('Could not find the project root.')

if str(current) not in sys.path:
    sys.path.insert(0, str(current))

import pandas as pd
import plotly.express as px

from src.data_loader import load_processed_data
from src.visualizations import create_multi_trend_chart

print('Imports ready.')

Imports ready.


### Interpretation

Setup ran cleanly. All analysis below uses the processed long dataset only.

## 1. Load and prepare

**Question:** what subset is comparable across indicators?

**Method:** load the long dataset and keep the numeric-scaled indicators (0–n), converting each score to **% of its scale** so a 0–4 question and the 0–100 total are comparable. The 1–7 ratings and the categorical STATUS are excluded here on purpose — they get their own sections.

In [2]:
sm_map = {'0_TO_4': 4, '0_TO_12': 12, '0_TO_16': 16, '0_TO_40': 40, '0_TO_60': 60, '0_TO_100': 100}

long = load_processed_data()
num = long[long['UNIT_MEASURE'].isin(sm_map)].copy()
num['Score'] = pd.to_numeric(num['Score'], errors='coerce')
num['pct'] = num['Score'] / num['UNIT_MEASURE'].map(sm_map) * 100

print('Numeric indicators:', num['INDICATOR'].nunique(), '| rows:', len(num))
print('Scales in use:', sorted(num['UNIT_MEASURE'].unique()))

Numeric indicators: 37 | rows: 102046
Scales in use: ['0_TO_100', '0_TO_12', '0_TO_16', '0_TO_4', '0_TO_40', '0_TO_60']


### Interpretation

37 indicators live on the six 0–n scales (the 40 minus the two ratings and the categorical status). Every level below is expressed as a share of the indicator's own scale, so questions, subtotals and totals are directly comparable.

## 2. The 40 indicators at a glance

**Question:** what does the full indicator profile look like — level and change together?

**Method:** for every indicator, take the 2026 global mean as % of scale and the 2013→2026 change, and print the dictionary-ordered profile table.

In [3]:
mean_2026 = num[num['Year'] == 2026].groupby(['INDICATOR', 'INDICATOR_LABEL', 'Category', 'UNIT_MEASURE'])['pct'].mean()
mean_2013 = num[num['Year'] == 2013].groupby('INDICATOR')['pct'].mean()

profile = mean_2026.reset_index().rename(columns={'pct': 'mean_pct_2026'})
profile['mean_pct_2013'] = profile['INDICATOR'].map(mean_2013)
profile['change_pct'] = (profile['mean_pct_2026'] - profile['mean_pct_2013']).round(1)
profile['mean_pct_2026'] = profile['mean_pct_2026'].round(1)
profile = profile.sort_values('mean_pct_2026')

print('Indicators with no non-missing 2026 scores (drop out of the ranking):')
print(profile[profile['mean_pct_2026'].isna()]['INDICATOR'].tolist())
print()
profile[['INDICATOR', 'Category', 'mean_pct_2026', 'change_pct']].to_string(index=False)

Indicators with no non-missing 2026 scores (drop out of the ranking):
['FH_FIW_ADD_A']



'   INDICATOR                                Category  mean_pct_2026  change_pct\nFH_FIW_ADD_Q Political rights, additional Question Q            3.7         1.4\n   FH_FIW_F4                             Rule of Law           45.3        -4.3\n   FH_FIW_C2               Functioning of Government           45.4        -3.2\n   FH_FIW_F2                             Rule of Law           47.6        -5.7\n   FH_FIW_C3               Functioning of Government           48.6        -2.4\n    FH_FIW_F                         Civil liberties           49.2        -3.1\n    FH_FIW_C                        Political rights           49.4        -4.2\n   FH_FIW_G4 Personal Autonomy And Individual Rights           50.8        -1.8\n   FH_FIW_F3                             Rule of Law           51.1        -1.5\n   FH_FIW_D1        Freedom of Expression and Belief           52.2        -5.2\n   FH_FIW_F1                             Rule of Law           52.8        -1.0\n   FH_FIW_C1               

### Interpretation

The profile table is the data dictionary in action: every indicator's level (2026) and direction (2013→2026) on one scale. Two things stand out already:

- **`FH_FIW_ADD_Q` sits at 3.7% of its scale** — an additional political-rights question that almost every economy fails — and is the only indicator that *improved* from an even lower base.
- **`FH_FIW_ADD_A` has no non-missing 2026 scores** and drops out of the 2026 ranking (a genuine missingness in the raw data, kept missing per policy).

## 3. Which dimensions are weakest and strongest?

**Question:** where does the world as a whole score worst and best in 2026?

**Method:** rank all 36 usable indicators by their 2026 global mean as % of scale.

In [4]:
ranking = profile.dropna(subset=['mean_pct_2026']).sort_values('mean_pct_2026')
fig = px.bar(
    ranking,
    x='mean_pct_2026',
    y='INDICATOR',
    orientation='h',
    color='Category',
    title='Global mean score as % of scale, all indicators, 2026',
    labels={'mean_pct_2026': '% of scale', 'INDICATOR': 'Indicator'},
)
fig.update_layout(template='plotly_white', title_x=0.5, height=800)
fig.show()

### Interpretation

The global ranking is led by **expression and personal-autonomy items** — `D2` (71.6%), `D3` (67.0%), `G1` (65.8%), `D4` (64.7%) — and anchored at the bottom by **rule of law and government function**: `F4` (45.3%), `C2` (45.4%), `F2` (47.6%), `C3` (48.6%), with the `F` and `C` subtotals both near 49%. Rule of law and the functioning of government are the world's weakest freedom dimensions, expression the strongest.

## 4. Change over time — the seven categories

**Question:** which dimensions deteriorated most since 2013?

**Method:** the seven category subtotals as global mean % of scale per year, one line each.

In [5]:
cats = ['FH_FIW_A', 'FH_FIW_B', 'FH_FIW_C', 'FH_FIW_D', 'FH_FIW_E', 'FH_FIW_F', 'FH_FIW_G']
cat_names = {'FH_FIW_A': 'A Electoral process', 'FH_FIW_B': 'B Pluralism & participation', 'FH_FIW_C': 'C Functioning of government', 'FH_FIW_D': 'D Expression & belief', 'FH_FIW_E': 'E Associational rights', 'FH_FIW_F': 'F Rule of law', 'FH_FIW_G': 'G Personal autonomy'}

cat_trend = num[num['INDICATOR'].isin(cats)].groupby(['INDICATOR', 'Year'])['pct'].mean().reset_index()
cat_trend['category'] = cat_trend['INDICATOR'].map(cat_names)

fig = create_multi_trend_chart(
    cat_trend,
    title='Global mean score as % of scale, by category, 2013-2026',
    y_label='Mean score as % of scale',
    group_col='category',
    value_col='pct',
)
fig.show()

### Interpretation

All seven categories declined — but unevenly. **Expression & belief (D) fell most, −6.5 points of scale** (70.3% → 63.8%), yet remains the strongest category; **electoral process (A) fell −6.1** (64.3% → 58.2%). At the other end, **personal autonomy (G) declined least (−1.6**, 60.5% → 58.9%) and rule of law (F) least after it (−3.1). The answer to the notebook's guiding question: the erosion is broad, but concentrated in **expression and the electoral process**, not in personal autonomy.

## 5. The 1–7 ratings — an inverted scale

**Question:** how are economies distributed on the classic 1–7 ratings?

**Method:** count the 2026 values of `FH_FIW_PR_RATING` and `FH_FIW_CL_RATING`.

**Important:** these two indicators are **inverted** relative to every other scale — **1 = most free, 7 = least free**. A rating of 7 is the worst possible. They are never mixed into the 0–n views.

In [6]:
for ind, label in [('FH_FIW_PR_RATING', 'Political rights rating'), ('FH_FIW_CL_RATING', 'Civil liberties rating')]:
    r = long[(long['INDICATOR'] == ind) & (long['Year'] == 2026)]['Score'].dropna().astype(float)
    counts = r.value_counts().sort_index()
    print(f'{label} 2026 (1 = most free, 7 = least free):')
    print(counts.to_string())
    print()

fig = px.bar(
    long[(long['INDICATOR'] == 'FH_FIW_PR_RATING') & (long['Year'] == 2026)]['Score'].dropna().astype(float).value_counts().sort_index(),
    title='Political rights rating 2026: number of economies at each rating',
    labels={'value': 'Number of economies', 'index': 'Rating (1 = most free)'},
    text_auto=True,
)
fig.update_layout(template='plotly_white', title_x=0.5)
fig.show()

Political rights rating 2026 (1 = most free, 7 = least free):
Score
1.0    46
2.0    38
3.0    21
4.0    16
5.0    14
6.0    19
7.0    42

Civil liberties rating 2026 (1 = most free, 7 = least free):
Score
1.0    44
2.0    34
3.0    24
4.0    28
5.0    27
6.0    25
7.0    14



### Interpretation

The political-rights rating is **bimodal**: 46 economies at rating 1 (most free) and 42 at rating 7 (least free), with a thin middle. Civil liberties are smoother (44 at 1, only 14 at 7). The world is polarizing into a best-rated and a worst-rated group, with fewer economies in between — matching the two-tier regional picture from notebook 06.

## 6. Rating movement since 2013

**Question:** are economies moving between ratings — and in which direction?

**Method:** for each economy, compare the PR rating in 2013 and 2026; a positive difference means the economy moved toward 7 (worse).

In [7]:
r13 = long[(long['INDICATOR'] == 'FH_FIW_PR_RATING') & (long['Year'] == 2013)].set_index('Economy')['Score'].astype(float)
r26 = long[(long['INDICATOR'] == 'FH_FIW_PR_RATING') & (long['Year'] == 2026)].set_index('Economy')['Score'].astype(float)
both = pd.concat([r13, r26], axis=1, keys=['r2013', 'r2026']).dropna()
both['diff'] = both['r2026'] - both['r2013']

print('Economies with both ratings:', len(both))
print('Improved (rating decreased, toward 1):', int((both['diff'] < 0).sum()))
print('Worsened (rating increased, toward 7):', int((both['diff'] > 0).sum()))
print('Unchanged:', int((both['diff'] == 0).sum()))
print('Mean rating change:', round(both['diff'].mean(), 2))

Economies with both ratings: 196
Improved (rating decreased, toward 1): 21
Worsened (rating increased, toward 7): 67
Unchanged: 108
Mean rating change: 0.35


### Interpretation

**67 economies were rated worse in 2026 than 2013, against 21 that improved** (108 unchanged) — a mean move of +0.35 on the inverted scale, i.e. toward less free. This is the ratings' view of the same decline the overall score showed in notebook 04: the two scales tell the same story.

## 7. The categorical status over time

**Question:** how has the Free / Partly Free / Not Free mix shifted since 2013?

**Method:** count the three statuses per year. STATUS is categorical — its strings are used directly.

In [8]:
status = long[long['INDICATOR'] == 'FH_FIW_STATUS']
st = status.pivot_table(index='Year', columns='Score', values='Economy', aggfunc='count')
print(st[['F', 'PF', 'NF']].to_string())

status_trend = st[['F', 'PF', 'NF']].reset_index().melt(id_vars='Year', var_name='status', value_name='count')
fig = create_multi_trend_chart(
    status_trend,
    title='Number of economies by status, 2013-2026',
    y_label='Number of economies',
    group_col='status',
    value_col='count',
)
fig.show()

Score   F  PF  NF
Year             
2013   91  59  47
2014   89  60  48
2015   90  56  51
2016   87  60  50
2017   87  60  49
2018   88  59  49
2019   86  60  50
2020   83  64  49
2021   82  60  54
2022   83  57  56
2023   84  55  57
2024   83  57  56
2025   85  52  59
2026   88  49  59


### Interpretation

The mix shifted steadily: **Not Free grew from 47 to 59 economies (23.9% → 30.1%)** while **Free fell from 91 to 88 (46.2% → 44.9%)** and Partly Free shrank from 59 to 49. The middle of the classification is being squeezed — economies are moving out of Partly Free into Not Free faster than anything moves into Free. This is the categorical footprint of the global decline.

## 8. Do the statuses separate cleanly by score?

**Question:** is the status classification consistent with the numeric overall score?

**Method:** box plots of the 2026 overall score grouped by status — a data-driven check that the three categories correspond to distinct score regions (no thresholds invented here; the data speaks).

In [9]:
tot = long[(long['INDICATOR'] == 'FH_FIW_TOTAL') & (long['Year'] == 2026)].copy()
tot['Score'] = pd.to_numeric(tot['Score'], errors='coerce')
m = tot.merge(status[status['Year'] == 2026][['Economy', 'Score']], on='Economy', suffixes=('', '_status'))

print(m.groupby('Score_status')['Score'].agg(['median', 'min', 'max']).to_string())

fig = px.box(
    m,
    x='Score_status',
    y='Score',
    title='Overall score 2026 by status classification',
    labels={'Score': 'Overall score (0-100)', 'Score_status': 'Status'},
)
fig.update_layout(template='plotly_white', title_x=0.5)
fig.show()

              median   min    max
Score_status                     
F               89.0  67.0  100.0
NF              17.0   0.0   33.0
PF              53.0  32.0   69.0


### Interpretation

The three statuses separate almost perfectly: **Free economies sit at a median of 89** (range 67–100), **Partly Free at 53** (32–69), **Not Free at 17** (0–33). The bands barely touch — only the Partly Free / Not Free boundary overlaps by a point or two. The categorical status and the numeric score are two consistent views of the same underlying classification, which validates both.

## 9. The extremes on the biggest-mover indicator

**Question:** which economies sit at the poles of `FH_FIW_D4` — the indicator that declined most (−12.9% of scale since 2013)?

**Method:** list the lowest- and highest-scoring economies on `D4` in 2026 (scale 0–4).

In [10]:
d4 = long[(long['INDICATOR'] == 'FH_FIW_D4') & (long['Year'] == 2026)][['Economy', 'Score']].dropna()
d4['Score'] = d4['Score'].astype(float)

print('Lowest 5 (score 0 of 4):')
print(d4.nsmallest(5, 'Score').to_string(index=False))
print()
print('Highest 5 (score 4 of 4):')
print(d4.nlargest(5, 'Score').to_string(index=False))

Lowest 5 (score 0 of 4):
                 Economy  Score
                  Rwanda    0.0
Central African Republic    0.0
                 Burundi    0.0
       Equatorial Guinea    0.0
      Russian Federation    0.0

Highest 5 (score 4 of 4):
         Economy  Score
          Latvia    4.0
 Slovak Republic    4.0
Papua New Guinea    4.0
          Norway    4.0
      San Marino    4.0


### Interpretation

On the freedom-of-expression question `D4` — the world's biggest decliner since 2013 — the 2026 extremes are stark: **Rwanda, the Central African Republic, Burundi, Equatorial Guinea and Russia score 0 of 4**, while **Latvia, the Slovak Republic, Papua New Guinea, Norway and San Marino score 4 of 4**. Note the geography: three of the five lowest are in the EAC's neighbourhood (Rwanda, Burundi — with Tanzania at 1), tying the indicator story back to notebooks 05–06.

## Summary and next question

### What we learned

- **Weakest dimensions (2026)**: rule of law (`F4`, `F2`) and government function (`C2`, `C3`) at ~45–49% of scale; **strongest**: expression items (`D2` 71.6%, `D3` 67.0%). The additional question `ADD_Q` sits at 3.7%; `ADD_A` has no 2026 data.
- **Biggest declines**: expression (−6.5) and electoral process (−6.1) at category level; `D4` (−12.9) at question level. Personal autonomy declined least (−1.6).
- **Ratings (1–7, inverted)**: PR ratings are bimodal — 46 economies at 1, 42 at 7; 67 economies were rated worse in 2026 than 2013 vs 21 better.
- **Status**: Not Free grew from 47 to 59 economies (23.9% → 30.1%); Partly Free is being squeezed out. Status separates almost perfectly by score (medians 89 / 53 / 17) — two consistent views.
- The ratings and STATUS are never coerced into numbers — they are analysed on their own terms.

### Next question

*How do these patterns look on a map?* — notebook 08, geographic analysis, plots the overall score and its components on interactive choropleth maps.